# Text Summarization Project - Google Colab Notebook
## End-to-End Text Summarization using Hugging Face Transformers

This notebook demonstrates how to use the text-summarizer project in Google Colab with full setup, training, evaluation, and inference capabilities.

## 1. Setup Colab Runtime (GPU/TPU) and Notebook Basics

**Important:** Go to **Runtime** → **Change runtime type** → Select **GPU** (or TPU)

Then run the cell below to verify GPU availability:

In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"CUDA Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"PyTorch Version: {torch.__version__}")

## 2. Install and Import Required Libraries

In [ ]:
# Install required packages
!pip install -q transformers datasets evaluate accelerate rouge-score sentencepiece tokenizers python-box pyyaml

In [ ]:
# Import libraries
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from datasets import load_dataset, Dataset
import evaluate
import json

print("All libraries imported successfully!")

## 3. Mount Google Drive and Configure Paths

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Set up paths
base_path = '/content/drive/MyDrive/text-summarizer'
checkpoint_dir = os.path.join(base_path, 'checkpoints')
output_dir = os.path.join(base_path, 'outputs')

# Create directories if they don't exist
os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

print(f"Base path: {base_path}")
print(f"Checkpoint directory: {checkpoint_dir}")
print(f"Output directory: {output_dir}")

## 4. Load and Inspect Your Dataset

In [ ]:
# Option 1: Load from a CSV file in Drive
# df = pd.read_csv('/content/drive/MyDrive/your-data.csv')
# print(df.head())

# Option 2: Load from Hugging Face datasets
dataset = load_dataset('cnn_dailymail', '3.0.0', split='train[:100]')  # Using a sample for demo

# Display dataset info
print(f"Dataset size: {len(dataset)}")
print(f"Columns: {dataset.column_names}")
print(f"\nSample entry:")
print(f"Article: {dataset[0]['article'][:200]}...")
print(f"\nSummary: {dataset[0]['highlights']}")

## 5. Text Preprocessing and Cleaning

In [ ]:
import re
from html.parser import HTMLParser

def clean_text(text):
    """
    Clean text data: remove URLs, HTML tags, extra whitespace
    """
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Test cleaning function
sample_text = dataset[0]['article']
cleaned_text = clean_text(sample_text)
print(f"Original length: {len(sample_text)}")
print(f"Cleaned length: {len(cleaned_text)}")
print(f"Cleaned text: {cleaned_text[:200]}...")

## 6. Tokenizer Setup and Examples

In [ ]:
# Load tokenizer
model_name = "facebook/bart-base"  # or "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Configure lengths
max_source_length = 1024
max_target_length = 128

# Test tokenization
sample_article = dataset[0]['article']
sample_summary = dataset[0]['highlights']

# Tokenize
source_tokens = tokenizer(sample_article, max_length=max_source_length, truncation=True, return_tensors='pt')
target_tokens = tokenizer(sample_summary, max_length=max_target_length, truncation=True, return_tensors='pt')

print(f"Source tokens shape: {source_tokens['input_ids'].shape}")
print(f"Target tokens shape: {target_tokens['input_ids'].shape}")
print(f"Tokenizer vocab size: {len(tokenizer)}")

## 7. Create Dataset and DataLoaders

In [ ]:
from torch.utils.data import DataLoader, Dataset as TorchDataset

class SummarizationDataset(TorchDataset):
    def __init__(self, dataset, tokenizer, max_source_length=1024, max_target_length=128):
        self.tokenizer = tokenizer
        self.max_source_length = max_source_length
        self.max_target_length = max_target_length
        self.data = dataset
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        article = self.data[idx]['article']
        summary = self.data[idx]['highlights']
        
        # Tokenize inputs
        inputs = self.tokenizer(
            article,
            max_length=self.max_source_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        
        # Tokenize targets
        labels = self.tokenizer(
            summary,
            max_length=self.max_target_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        
        return {
            'input_ids': inputs['input_ids'].squeeze(),
            'attention_mask': inputs['attention_mask'].squeeze(),
            'labels': labels['input_ids'].squeeze()
        }

# Create dataset and dataloader
train_dataset = SummarizationDataset(dataset, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

print(f"Dataset created with {len(train_dataset)} samples")
print(f"DataLoader created with batch size 4")

## 8. Load Pretrained Model

In [ ]:
# Load model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

# Print model size
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model: {model_name}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Device: {device}")

## 9. Prepare Training Loop

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import StepLR

# Training configuration
learning_rate = 1e-4
num_epochs = 2
gradient_accumulation_steps = 2

# Optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=learning_rate)
scheduler = StepLR(optimizer, step_size=1, gamma=0.1)

print(f"Learning rate: {learning_rate}")
print(f"Epochs: {num_epochs}")
print(f"Gradient accumulation steps: {gradient_accumulation_steps}")

## 10. Fine-tuning on Colab (with Checkpointing)

In [ ]:
def train_epoch(model, train_loader, optimizer, device, gradient_accumulation_steps=2):
    model.train()
    total_loss = 0
    
    for batch_idx, batch in enumerate(tqdm(train_loader)):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss / gradient_accumulation_steps
        
        # Backward pass
        loss.backward()
        
        # Optimizer step
        if (batch_idx + 1) % gradient_accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
        
        total_loss += outputs.loss.item()
    
    avg_loss = total_loss / len(train_loader)
    return avg_loss

# Start training
print("Starting training...")
for epoch in range(num_epochs):
    avg_loss = train_epoch(model, train_loader, optimizer, device, gradient_accumulation_steps)
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}")
    
    # Save checkpoint
    checkpoint_path = os.path.join(checkpoint_dir, f'checkpoint_epoch_{epoch+1}')
    model.save_pretrained(checkpoint_path)
    tokenizer.save_pretrained(checkpoint_path)
    print(f"Checkpoint saved to {checkpoint_path}")
    
    scheduler.step()

print("Training completed!")

## 11. Inference and Generation

In [ ]:
def generate_summary(text, model, tokenizer, device, max_length=128):
    """
    Generate summary for input text
    """
    # Prepare input
    inputs = tokenizer.encode(text, return_tensors='pt', max_length=1024, truncation=True).to(device)
    
    # Generate summary
    summary_ids = model.generate(
        inputs,
        max_length=max_length,
        min_length=50,
        num_beams=4,
        early_stopping=True,
        length_penalty=2.0
    )
    
    # Decode summary
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

# Test inference
test_article = dataset[0]['article']
generated_summary = generate_summary(test_article, model, tokenizer, device)

print("Original Article:")
print(test_article[:300] + "...")
print("\nGenerated Summary:")
print(generated_summary)
print("\nReference Summary:")
print(dataset[0]['highlights'])

## 12. Evaluation (ROUGE Scores)

In [ ]:
# Load ROUGE metric
rouge = evaluate.load('rouge')

def evaluate_model(model, dataset, tokenizer, device, num_samples=10):
    """
    Evaluate model on a subset of the dataset
    """
    predictions = []
    references = []
    
    model.eval()
    with torch.no_grad():
        for idx in tqdm(range(min(num_samples, len(dataset)))):
            article = dataset[idx]['article']
            reference = dataset[idx]['highlights']
            
            prediction = generate_summary(article, model, tokenizer, device)
            predictions.append(prediction)
            references.append(reference)
    
    # Calculate ROUGE scores
    results = rouge.compute(predictions=predictions, references=references)
    return results

# Evaluate on a small subset
print("Evaluating model...")
eval_results = evaluate_model(model, dataset, tokenizer, device, num_samples=5)

print("\nEvaluation Results:")
for metric, score in eval_results.items():
    print(f"{metric}: {score:.4f}")

## 13. Save/Export Model and Artifacts

In [ ]:
# Save final model and tokenizer
final_model_path = os.path.join(output_dir, 'final_model')
model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

print(f"Model saved to {final_model_path}")

# Save evaluation results
results_path = os.path.join(output_dir, 'evaluation_results.json')
with open(results_path, 'w') as f:
    json.dump(eval_results, f, indent=2)

print(f"Results saved to {results_path}")

# Optional: Push to Hugging Face Hub
# model.push_to_hub('your-username/text-summarizer')
# tokenizer.push_to_hub('your-username/text-summarizer')

## 14. Performance Tips for Colab

### Memory Optimization Tips:
1. **Use smaller models**: `facebook/bart-base` instead of `facebook/bart-large`
2. **Enable mixed precision**: Use `fp16=True` in training arguments
3. **Gradient accumulation**: Accumulate gradients before optimizer step
4. **Smaller batch sizes**: Start with batch_size=2 or 4
5. **Shorter max lengths**: Reduce `max_source_length` and `max_target_length`
6. **Clear cache**: Use `torch.cuda.empty_cache()` between batches

### Runtime Limits:
- Free tier: 12 hours max
- Colab Pro: 24 hours max
- Save checkpoints frequently to avoid losing progress
- Use Colab Pro or Pro+ for longer training runs

In [ ]:
# Memory optimization utilities
def clear_memory():
    torch.cuda.empty_cache()
    print("Memory cleared")

def get_memory_usage():
    if torch.cuda.is_available():
        print(f"GPU Memory Used: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
        print(f"GPU Memory Reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
    else:
        print("CUDA not available")

# Check memory
get_memory_usage()

## 15. Reproducibility and Logging

In [ ]:
import random

def set_seed(seed=42):
    """
    Set random seeds for reproducibility
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Random seed set to {seed}")

# Set seed at start
set_seed(42)

# Log configuration
config = {
    'model_name': model_name,
    'learning_rate': learning_rate,
    'num_epochs': num_epochs,
    'batch_size': 4,
    'max_source_length': 1024,
    'max_target_length': 128,
    'seed': 42
}

config_path = os.path.join(output_dir, 'config.json')
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f"Configuration saved to {config_path}")

## Next Steps

1. **Customize the dataset**: Replace with your own data
2. **Adjust hyperparameters**: Modify learning rate, batch size, epochs
3. **Try different models**: Experiment with T5, BART, or other seq2seq models
4. **Monitor training**: Add validation and early stopping
5. **Push to Hub**: Share your trained model on Hugging Face Hub
6. **Deploy**: Use your model with the FastAPI app for inference

### Useful Resources:
- [Hugging Face Documentation](https://huggingface.co/docs)
- [Transformers Examples](https://github.com/huggingface/transformers/tree/main/examples)
- [ROUGE Score Explanation](https://en.wikipedia.org/wiki/ROUGE_(metric))